# Gradient Descent

In the previous notebooks you called `LinearRegression().fit(X, y)` and scikit-learn handed
back a slope and an intercept. It never said how it found them.

This notebook shows one way to find them. You will write, from scratch, **gradient descent**, the algorithm
that trains most machine learning models with a differentiable loss, including every neural network.
The model here is the simplest one you know, a straight line, but the method is the same one used
to train neural networks with billions of parameters. (`LinearRegression` itself takes a shortcut,
the direct least squares solution in the bonus section at the end.)

> [!TIP]
> This notebook is an optional exercise. Work through it if you have time left after the other notebooks, or if you want to see what happens inside `LinearRegression().fit()`.

## Learning Objectives

At the end of this notebook, you should be able to:

- Implement a loss function that scores how badly a line fits the data, and estimate its gradient numerically.
- Implement the gradient descent loop and use it to recover a known slope and intercept.
- Compare how different learning rates decide whether gradient descent converges or diverges.

## The Plan

You will build gradient descent in four steps. Each step is a small function that the next step uses:

<img src="assets/gradient_descent_steps.png" width="800">

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Fixing the seed keeps the generated data identical every run,
# so your numbers match the ones discussed here.
np.random.seed(42)

plt.rcParams["figure.figsize"] = (7, 4)

## The data

We generate our own data, because then we know the right answer. A straight line has two
parameters, as in notebook 01:

$$y = b_0 + b_1 x$$

- $b_1$ is the **slope**: how much $y$ changes when $x$ increases by 1.
- $b_0$ is the **intercept**: the value of $y$ when $x = 0$.

The points below sit on the line with $b_1 = 2.0$ and $b_0 = -1.5$,

$$y = -1.5 + 2.0x$$

with a little random noise added, so the task is realistic. **Gradient descent should
recover a slope near 2.0 and an intercept near -1.5**, and that is how you will know it
worked.

NumPy is used only to generate the random numbers. The data is stored in pandas Series, so
everything you write from here on works on Series, just like the columns of a DataFrame.

In [ ]:
TRUE_SLOPE = 2.0
TRUE_INTERCEPT = -1.5

# 200 x values between 0 and 1, then the matching y values plus Gaussian noise.
X = pd.Series(np.random.rand(200))
noise = np.random.normal(0, 0.2, 200)
y = X * TRUE_SLOPE + TRUE_INTERCEPT + noise

plt.scatter(X, y, s=8)
plt.xlabel("x")
plt.ylabel("y")
plt.title("The data we will fit a line to");

## Step 1: a function that draws a line

A line has two parameters, a **slope** and an **intercept**. Given both, together with the x
values, you can compute the predicted y values.

Write `make_line()` so it returns the predicted y values.

In [ ]:
def make_line(xdata, slope, intercept):
    """
    Compute the y values of a straight line.

    Parameters
    ----------
    xdata : pandas Series
        The x values to evaluate the line at.
    slope : float
        The slope of the line.
    intercept : float
        The y value where the line crosses x = 0.

    Returns
    -------
    pandas Series
        The predicted y value for each x.
    """
    # TODO: return the predicted y values for this slope and intercept
    ypred = ...

    return ypred

## Step 2: a function that scores the line

You can look at a plot and say whether a line fits well. The computer cannot, so it needs a
number. That number is the **loss**.

We use the **mean squared error** (MSE). It is computed in three steps:

- Take the gap between each true y and its predicted y.
- Square each gap, so that positive and negative gaps both count as error.
- Average the squared gaps.

$$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

A **lower** MSE means a better line. Training the model means making this number as small as
possible.

In [ ]:
def mse(ytrue, ypred):
    """
    Compute the mean squared error between true and predicted values.

    This is our loss function: the quantity gradient descent will minimise.

    Parameters
    ----------
    ytrue : pandas Series
        The observed y values.
    ypred : pandas Series
        The y values predicted by the line.

    Returns
    -------
    float
        The mean of the squared differences.
    """
    # TODO: return the mean of the squared differences between ytrue and ypred
    error = ...

    return error

### Check: a good line and a bad one

If your two functions work, the good line should score a much lower MSE than the bad one.

In [ ]:
good = make_line(X, slope=2.0, intercept=-1.5)
bad = make_line(X, slope=4.0, intercept=0.0)

print(f"MSE of the good line: {mse(y, good):.4f}")
print(f"MSE of the bad line:  {mse(y, bad):.4f}")

plt.scatter(X, y, s=8, label="data")
plt.plot(X, good, color="green", label="good line")
plt.plot(X, bad, color="red", label="bad line")
plt.legend();

## Step 3: which way is downhill?

We have a score. Now we need a direction: should the slope go up or down to make the loss
smaller? And the intercept?

The exact answer comes from calculus, but you can get the same information without it. **Increase one
parameter by a tiny amount $h$ and see whether the loss rises or falls.** The change in loss
divided by $h$ is the **gradient** for that parameter.

- Gradient **positive**: increasing the parameter increased the loss, so go down.
- Gradient **negative**: increasing the parameter decreased the loss, so go up.

In symbols, with the tiny change $h$ (the `h` argument in the code):

$$
\frac{\partial MSE}{\partial b_1} \approx \frac{MSE(b_1 + h,\ b_0) - MSE(b_1,\ b_0)}{h}
\qquad
\frac{\partial MSE}{\partial b_0} \approx \frac{MSE(b_1,\ b_0 + h) - MSE(b_1,\ b_0)}{h}
$$

In the code, each term of the formula has its own variable:

| Formula | Variable in `calc_gradient()` |
|---|---|
| $MSE(b_1,\ b_0)$ | `loss_now` |
| $MSE(b_1 + h,\ b_0)$ | `loss_slope_plus_h` |
| $MSE(b_1,\ b_0 + h)$ | `loss_intercept_plus_h` |
| $\partial MSE / \partial b_1$ | `deriv_slope` |
| $\partial MSE / \partial b_0$ | `deriv_intercept` |

The symbol $\partial$ reads "the partial derivative": how much the loss changes when only that one parameter changes.

<details><summary>For the curious: the exact gradient</summary>

With calculus, the two partial derivatives of the MSE have an exact form:

$$
\frac{\partial MSE}{\partial b_1} = -\frac{2}{n}\sum_{i=1}^{n} x_i\,(y_i - \hat{y}_i)
\qquad
\frac{\partial MSE}{\partial b_0} = -\frac{2}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)
$$

At the starting guess used below, your estimates and these exact values agree to about three decimal places.

</details>

Do this once for the slope and once for the intercept.

In [ ]:
def calc_gradient(xdata, ytrue, slope, intercept, h=0.0001):
    """
    Estimate the gradient of the MSE for the slope (b1) and the intercept (b0).

    For each parameter in turn, add the tiny change h, recompute the MSE,
    and divide the change in MSE by h:

        ∂MSE/∂b1 ≈ (MSE(b1 + h, b0) - MSE(b1, b0)) / h
        ∂MSE/∂b0 ≈ (MSE(b1, b0 + h) - MSE(b1, b0)) / h

    No calculus required.

    Parameters
    ----------
    xdata : pandas Series
        The x values.
    ytrue : pandas Series
        The observed y values.
    slope : float
        The current slope, b1.
    intercept : float
        The current intercept, b0.
    h : float, optional
        The tiny change added to one parameter at a time.

    Returns
    -------
    tuple of float
        The gradient for the slope (∂MSE/∂b1) and for the intercept (∂MSE/∂b0).
    """
    # MSE(b1, b0): the loss at the current slope and intercept
    loss_now = mse(ytrue, make_line(xdata, slope, intercept))

    # TODO: compute MSE(b1 + h, b0), the loss after increasing only the slope by h
    loss_slope_plus_h = ...
    # TODO: compute ∂MSE/∂b1 ≈ (MSE(b1 + h, b0) - MSE(b1, b0)) / h
    deriv_slope = ...

    # TODO: compute MSE(b1, b0 + h), the loss after increasing only the intercept by h
    loss_intercept_plus_h = ...
    # TODO: compute ∂MSE/∂b0 ≈ (MSE(b1, b0 + h) - MSE(b1, b0)) / h
    deriv_intercept = ...

    return deriv_slope, deriv_intercept

## Step 4: walk downhill

Now the algorithm itself. Starting from a deliberately bad guess, repeat:

1. Compute the gradient for both parameters.
2. If the gradient is tiny, we are at the bottom, so stop.
3. Otherwise move **against** the gradient, by an amount set by the **learning rate**.

As a formula, with $\eta$ (the Greek letter *eta*) for the learning rate, each step updates both parameters:

$$
b_1 \leftarrow b_1 - \eta\,\frac{\partial MSE}{\partial b_1}
\qquad
b_0 \leftarrow b_0 - \eta\,\frac{\partial MSE}{\partial b_0}
$$

The arrow $\leftarrow$ means "replace with". The minus sign makes the step go *against* the gradient, downhill. The loop stops once

$$
\left|\frac{\partial MSE}{\partial b_1}\right| + \left|\frac{\partial MSE}{\partial b_0}\right| \le \text{THRESHOLD}
$$

or after `MAX_ITERATIONS` steps.

<img src="assets/gradient_descent_loop.png" width="450">

The learning rate controls the step size. If it is too small, the parameters move towards
the minimum very slowly. If it is too large, each step overshoots the minimum and the
parameters bounce around it.

In [ ]:
# A deliberately bad starting guess: the true values are 2.0 and -1.5
slope = -8.0
intercept = 2.0

LEARNING_RATE = 0.5
MAX_ITERATIONS = 300
THRESHOLD = 0.0001  # stop once the gradient is this flat

loss_history = []

for i in range(MAX_ITERATIONS):
    # TODO: get the gradient for the current slope and intercept
    deriv_slope, deriv_intercept = ...

    loss_history.append(mse(y, make_line(X, slope, intercept)))

    # Stop early if the ground is flat enough
    if abs(deriv_slope) + abs(deriv_intercept) <= THRESHOLD:
        print(f"Converged after {i} iterations.")
        break

    # TODO: move each parameter against its gradient, scaled by the learning rate
    slope = ...
    intercept = ...
else:
    print(f"Stopped at the limit of {MAX_ITERATIONS} iterations without converging.")

print(f"slope:      {slope:.3f}  (true value {TRUE_SLOPE})")
print(f"intercept:  {intercept:.3f}  (true value {TRUE_INTERCEPT})")
print(f"final MSE:  {mse(y, make_line(X, slope, intercept)):.4f}")

## Did it work?

Check two things:

- The recovered parameters should be close to the ones we generated the data with.
- The loss should fall steeply and then flatten.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.scatter(X, y, s=8, label="data")
ax1.plot(X, make_line(X, slope, intercept), color="magenta", label="gradient descent")
ax1.set_title("The fitted line")
ax1.legend()

ax2.plot(loss_history)
ax2.set_xlabel("iteration")
ax2.set_ylabel("MSE")
ax2.set_title("Loss falling over time");

The loss drops fast at first and then barely moves. That flat tail is why the threshold
check is worth having: once the gradient is almost flat, further steps barely lower the loss, so
the loop stops after 138 iterations instead of running all 300.

## Your turn

1. Set `LEARNING_RATE` to `0.01` and rerun. Does it converge within 300 iterations, and how
   close does the slope get?
2. Set it to `2.0`. The loop announces that it converged, and the slope it reports is about
   200 billion. Look at the final MSE, then work out why the stopping rule was fooled.
   **A stopping rule that only checks the gradient cannot tell a flat valley from an
   explosion.**
3. Put the learning rate back to `0.5` and start from `slope = 100`. Does it still find its
   way back?

## Summary

In this notebook you:

- Implemented a loss function, the mean squared error, and estimated its gradient numerically. Training a model means **minimising** this loss.
- Implemented the gradient descent loop and recovered the slope and intercept the data was generated with.
- Compared how the **learning rate** changes convergence: too small is slow, too large diverges.

`LinearRegression().fit()` does this work for you. At a much larger scale, gradient descent also trains every neural network you will meet later in the course.

## Bonus: the Normal Equation

This section is optional. For linear regression there is also a direct route to the best line, without any walking downhill. Setting the gradient of the MSE to zero and solving for the parameters gives the best line in a single calculation, the **normal equation**:

$$
\hat{b} = (X^T X)^{-1} X^T y
$$

Here $X$ is the feature matrix with an extra column of ones, so the intercept is estimated together with the slope. This closed-form solution is what ordinary least squares computes. (scikit-learn's `LinearRegression` solves the same least squares problem, with a numerically more stable method than an explicit matrix inverse.)

The cell below is complete, so you can run it even before your gradient descent works.

In [ ]:
# Add a column of ones so the intercept is estimated together with the slope
X_design = pd.DataFrame({"intercept": 1.0, "x": X})

# Solve (X^T X) b = X^T y. np.linalg.solve is more stable than inverting the matrix.
intercept_ne, slope_ne = np.linalg.solve(X_design.T @ X_design, X_design.T @ y)

print(f"normal equation:  slope {slope_ne:.3f}, intercept {intercept_ne:.3f}")
print(f"true values:      slope {TRUE_SLOPE}, intercept {TRUE_INTERCEPT}")

The normal equation gives a slope of 1.984 and an intercept of -1.479, the same values gradient descent reached. Neither method recovers exactly 2.0 and -1.5, because the noise we added shifts the best-fitting line slightly.

| | Gradient descent | Normal equation |
|---|---|---|
| How it works | Many small steps downhill | One calculation |
| Needs a learning rate | Yes | No |
| With many features | Stays manageable | Solving the $p \times p$ system becomes slow when the number of features $p$ runs into the thousands |
| Works for other models | Yes, for almost any model with a differentiable loss | No, only for linear regression with the MSE loss |

So why learn gradient descent at all? The normal equation exists for this one model only. Logistic regression, neural networks and most other models have no closed-form solution, and gradient descent is how they are trained.